## 1. `bam_to_cz` vs `bam_to_allc`

In [1]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/test_cytozip"))

In [2]:
from ALLCools._bam_to_allc import bam_to_allc
%time bam_to_allc(bam_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  reference_fasta=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.fa"), \
                  output_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz", \
                  chroms=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes"))

[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai
[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai


CPU times: user 2min 40s, sys: 13.5 s, total: 2min 54s
Wall time: 3min 9s


,mc,cov,mc_rate,genome_cov
CTT,42937,2689050,0.015967,0.040206
CCT,25755,2274939,0.011321,0.040206
CAC,307965,2013638,0.152940,0.040206
CAG,159326,2715612,0.058670,0.040206
CCA,27004,2462628,0.010966,0.040206
CTG,40374,2644319,0.015268,0.040206
CCC,20390,1641759,0.012420,0.040206
CGC,244843,293296,0.834798,0.040206
CCG,5368,354632,0.015137,0.040206
CAA,111157,2702079,0.041138,0.040206


In [6]:
from cytozip.bam import bam_to_cz
%time bam_to_cz(bam_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  genome=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.fa"), \
                  output="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz", \
                  reference="~/Ref/hg38/hg38_with_chrL.allc.cz",\
                   chroms=os.path.expanduser("~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes") \
               ) 
# CPU times: user 1min 21s, sys: 7.41 s, total: 1min 29s
# Wall time: 1min 31s

[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai


CPU times: user 1min 19s, sys: 7.18 s, total: 1min 27s
Wall time: 1min 29s


,mc,cov,mc_rate,genome_cov
CTT,42937,2689050,0.015967,0.008107
CCT,25755,2274939,0.011321,0.008107
CAC,307965,2013638,0.152940,0.008107
CAG,159326,2715612,0.058670,0.008107
CCA,27004,2462628,0.010966,0.008107
CTG,40374,2644319,0.015268,0.008107
CCC,20390,1641759,0.012420,0.008107
CGC,244843,293296,0.834798,0.008107
CCG,5368,354632,0.015137,0.008107
CAA,111157,2702079,0.041138,0.008107


In [7]:
# validate whether cz and allc store the same values
df_allc=pd.read_csv("cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz", 
                    sep="\t", header=None, usecols=[0, 1,2,3,4, 5],names=["chrom", "pos", 'strand','context',"mc_allc", "cov_allc"],
    )
df_allc.set_index(['chrom','pos','strand','context'],inplace=True)
df_allc.head()

mc_allc  cov_allc
chrom pos   strand context                   
chr1  14932 -      CTT            0         1
      14933 -      CCT            0         1
      14935 -      CAC            1         1
      14938 -      CAG            1         1
      14939 -      CCA            0         1

In [8]:
from cytozip import Reader
cell = Reader("cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz")
df_cz=cell.to_df(reference="~/Ref/hg38/hg38_with_chrL.allc.cz")
df_cz.rename(columns={'mc':'mc_cz','cov':'cov_cz'},inplace=True)
df_cz.set_index(['chrom','pos','strand','context'],inplace=True)
df_cz

mc_cz  cov_cz
chrom pos   strand context               
chr1  14932 -      CTT          0       1
      14933 -      CCT          0       1
      14935 -      CAC          1       1
      14938 -      CAG          1       1
      14939 -      CCA          0       1
...                           ...     ...
chrL  48403 -      CAA          0       1
      48411 -      CAA          0       1
      48414 -      CGT          1       1
      48416 -      CAC          0       1
      48417 -      CCA          0       1

[26017334 rows x 2 columns]

In [14]:
df_combined=pd.concat([df_allc,df_cz],axis=1)
print(df_combined.head())
df_diff=df_combined.loc[(df_combined.mc_allc != df_combined.mc_cz) | (df_combined.cov_allc != df_combined.cov_cz)]
print(f"{df_diff.shape[0]} mismatch between allc and cz")
df_diff

                            mc_allc  cov_allc  mc_cz  cov_cz
chrom pos   strand context                                  
chr1  14932 -      CTT            0         1      0       1
      14933 -      CCT            0         1      0       1
      14935 -      CAC            1         1      1       1
      14938 -      CAG            1         1      1       1
      14939 -      CCA            0         1      0       1
0 mismatch


,,,,mc_allc,cov_allc,mc_cz,cov_cz
chrom,pos,strand,context,,,,


<a href="benchmark/benchmark_methylation_calling.ipynb">
    <img src="benchmark/bam_benchmark.png" title="kwargs, gap and pad" align="center" width="250px">
</a>